In [2]:
import numpy as np
from numpy.linalg import norm
import math, random
import polyscope as ps
import fcpw
import argparse
from pathlib import Path
from typing import Callable, Optional
import trimesh

In [3]:
def load_obj(obj_file_path):
    positions = []
    indices = []

    with open(obj_file_path, 'r') as file:
        for line in file:
            if line.startswith('v '):
                position = list(map(float, line.strip().split()[1:]))
                positions.append(np.array(position, dtype=np.float32, order='C'))

            elif line.startswith('f '):
                index = [int(idx.split('/')[0]) - 1 for idx in line.strip().split()[1:]]
                indices.append(np.array(index, dtype=np.int32, order='C'))

    return np.array(positions), np.array(indices)

def load_fcpw_scene(positions, indices, build_vectorized_cpu_bvh):
    # load positions and indices
    scene = fcpw.scene_3D()
    scene.set_object_count(1)
    scene.set_object_vertices(positions, 0)
    scene.set_object_triangles(indices, 0)

    # build scene on CPU
    aggregate_type = fcpw.aggregate_type.bvh_surface_area
    print_stats = False
    reduce_memory_footprint = False
    scene.build(aggregate_type, build_vectorized_cpu_bvh,
                print_stats, reduce_memory_footprint)

    return scene


In [4]:
class PDEenv:
    def __init__(self, scene, grid):
        self.scene = scene
        #### the shape inside the [-1, 1]^3 cube
        self.grid = grid
        self.eps = 0.01
        self.maxSteps = 50
        self.maxR = 10
    
    def perform_closest_point_queries(self, query_points):
        # perform cpqs
        squared_max_radii = np.inf * np.ones(len(query_points), dtype=np.float32)
        interactions = fcpw.interaction_3D_list()
        self.scene.find_closest_points(query_points, squared_max_radii, interactions)

        # extract closest points
        closest_points = np.array([i.p for i in interactions])

        return closest_points
    
    def generate_starting_points(self, pathAmount):
        #target = np.random.rand(pathAmount, 3).astype(np.float32)
        #target = 2* target - 1
        #mask = self.mesh.contains(target)
        #filtered_target = target[mask]
        #print(mask)
        #print(filtered_target)
        #return target
        target = self.grid.copy()
        np.random.shuffle(target)
        sample = target[:pathAmount]
        #print(sample)
        return np.array(sample)

        

    def shortestDistance(self, cur_points):
        closest_points = self.perform_closest_point_queries(cur_points)
        distance = norm(cur_points - closest_points, axis = 1)
        return distance
    
    def ball_Surface_sampling(self):
        uv = np.zeros((2))
        i = norm(uv)**2
        while i > 1 or i == 0:
            uv = (np.random.rand(2) - 0.5)* 2
            i = norm(uv)**2
            
        #print(direction)
        direction = np.zeros((3))
        direction[0] = 2 * uv[0] * math.sqrt(1 - i)
        direction[1] = 2 * uv[1] * math.sqrt(1 - i)
        direction[2] = 1 - 2*i
        return direction
    
    def next_step(self, cur_points):
        distance = self.shortestDistance(cur_points)
        if max(distance) < self.eps:
            return[]
        
        new_points = np.ones((len(cur_points), 3))
        for i in range(len(cur_points)):
            if distance[i] < self.eps:
                new_points[i] = cur_points[i]
            elif distance[i] < self.maxR:
                direction = self.ball_Surface_sampling()
                new_points[i] = cur_points[i] + distance[i] * direction
            else: 
                direction = self.ball_Surface_sampling()
                new_points[i] = cur_points[i] + self.maxR * direction
                
        return new_points
            
        
    def pathGenerator(self, pathAmount): 
        #### Here pathAmount is the starting point amount! Including the invalid paths generated!
        #### Get rid of the invalid paths later in this func, total generated len(path) <= pathAmount
        x0 = self.generate_starting_points(pathAmount)
        paths = []
        for step in range(0, self.maxSteps): 
            paths.append(x0)
            next_step = self.next_step(x0)
            if len(next_step) == 0:
                paths = np.swapaxes(paths,0,1)
                return np.array(paths)
            x0 = next_step
        #### Otherwise, remove the paths that exceed the maxSteps
        paths = np.swapaxes(paths,0,1)
        paths1 = []
        distance = self.shortestDistance(paths[:, self.maxSteps - 1, :].reshape(pathAmount, 3))
        for i in range(pathAmount):
            if distance[i] < self.eps:
                paths1.append(paths[i, :, :])
                
        print(len(paths1))
        return np.array(paths1)
    

In [5]:
#### Path Generation
positions, indices = load_obj("dragon1.obj")
    # load fcpw scene
scene = load_fcpw_scene(positions, indices, True)
mesh = trimesh.load("dragon1.obj")
vox = mesh.voxelized(pitch=0.02)

    # fill interior
vox_filled = vox.fill()
points = vox_filled.points
#mask = mesh.contains(points)
#grid_points= np.array(points[mask])
print(points.size)


204132


In [6]:
cur_PDE = PDEenv(scene, points)
path = cur_PDE.pathGenerator(50000)

49978


In [7]:
#ps.init()
#ps.set_ground_plane_mode("none")

# register mesh and callback
#ps.register_surface_mesh("mesh", positions, indices)
#path_positions = np.array(path).reshape(len(path)*len(path[0]), 3)
#path_indices = []
#for j in range(len(path)):
#    for i in range(len(path[0]) - 1):
#        path_indices.append(np.array([j * len(path[0]) + i, j * len(path[0]) + i + 1]))
    
#network = ps.register_curve_network("edges", path_positions, np.array(path_indices))
#network.set_radius(0.005, relative=False)
#ps.show()

In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import initializers
import time

In [9]:
#### Boundary generation for paths
def Boundary_Genration(path, g):
    endPoints = path[:, len(path[0]) - 1, :].reshape(len(path), 3)
    #print(endPoints)
    boundary = np.apply_along_axis(g, 1, endPoints)
    return boundary

In [10]:
def g(v):
    if max(abs(v[0]), abs(v[1]), abs(v[2])) > 0.98:
        return -1
    
    return v[0] + v[1] + v[2]

boundary = Boundary_Genration(path, g)

In [11]:
class NN(keras.Model):
    def __init__(self, units):
        super(NN, self).__init__()
        self.layer1 = keras.layers.Dense(units[0])
        self.layer2 = keras.layers.Dense(units[1])
        self.layer3 = keras.layers.Dense(units[2])
        self.outputs = layers.Dense(4)

    def call(self, input_tensor, training):
        x = self.layer1(input_tensor)
        x = tf.nn.relu(x)
        x = self.layer2(x)
        x = tf.nn.relu(x)
        x = self.layer3(x)
        x = tf.nn.relu(x)
        x = self.outputs(x)
        #print(x.shape)
        return x

In [12]:
class Ysolver(keras.Model):
    # set y as a trainable variable, initialized randomly
    def __init__(self, layer):
        super(Ysolver, self).__init__()
        self.model = NN(layer)

        #self.y = tf.Variable(np.random.uniform(low=-1, high=1, size=[1, 1]),trainable = True, dtype ="float32")

    def call(self, path, training, batch):
        #print(path.shape)
        y_pred = self.model(path[:, 0], training = training)[:, 0]
        #print(y_pred.shape)
        for i in range(len(path[0]) - 1): ###input path should be (-1, 2) shaped nparray, no len()
            #y = self.Ymodel(path[:, i], training)
            #print(y.shape)
            #z_input = tf.concat([path[:, i],y], 1)
            z = self.model(path[:, i], training = training)[:, 1:4] #obtain gradients for steps
            diff = path[:, i + 1] - path[:, i]
            #print(diff.shape)
            add = z[:, 0]*diff[:, 0] + z[:, 1]*diff[:, 1] + z[:, 2]*diff[:, 2] #accumulating
            #print(add.shape)
            y_pred += add

        return y_pred

In [13]:
class NNsolver(keras.Model):
    def __init__(self, paths, boundary):
        super(NNsolver, self).__init__()
        self.paths = paths
        self.boundary = np.array(boundary)
        self.model = Ysolver([64, 128, 128])
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4 , epsilon=1e-8)

    def loss_fn(self, y_pred, y):
        delta = y_pred - y
        loss = tf.reduce_mean(tf.square(delta))
        return loss

    def train_step(self, cur_path, training, batch_size):           
        return self.model(cur_path, training = training, batch = batch_size)

    def train(self, batch_size): 
        training_history = []
        for i in range((len(self.paths) - 1)//batch_size + 1):
            batch = batch_size
            if (i + 1) * batch_size > len(self.paths):
                batch = len(self.paths) - i * batch_size
            cur_path = self.paths[i*batch_size: i*batch_size + batch]
            cur_bound = self.boundary[i*batch_size: i*batch_size + batch]
                
            with tf.GradientTape() as tape:
                y_pred = self.train_step(cur_path, training = True, batch_size=batch)
                loss = self.loss_fn(y_pred, cur_bound)

            training_vars = self.model.trainable_variables 
            grad = tape.gradient(loss, training_vars)
            #print(grad)
            #print("----------------------------------------------------")
            self.optimizer.apply_gradients(zip(grad, training_vars))
            #print(self.model.Ymodel.trainable_variables)
            training_history.append(loss)

        return training_history

    def train_epoch(self, epoch, batch_size):
        for i in range(epoch):
            cur_history = self.train(batch_size)
            print("epoch: " + str(i) + " loss: "+ str(np.mean(np.array(cur_history))))


In [14]:
training = NNsolver(path, boundary)
training.train_epoch(50, 1024)

epoch: 0 loss: 0.30231497
epoch: 1 loss: 0.057001665
epoch: 2 loss: 0.010490581
epoch: 3 loss: 0.004677481
epoch: 4 loss: 0.0024917733
epoch: 5 loss: 0.001667499
epoch: 6 loss: 0.0013246626
epoch: 7 loss: 0.0011275979
epoch: 8 loss: 0.0009962169
epoch: 9 loss: 0.0009029774
epoch: 10 loss: 0.0008341618
epoch: 11 loss: 0.000780878
epoch: 12 loss: 0.0007380257
epoch: 13 loss: 0.000700974
epoch: 14 loss: 0.00067016465
epoch: 15 loss: 0.0006431445
epoch: 16 loss: 0.0006190424
epoch: 17 loss: 0.00059695815
epoch: 18 loss: 0.0005776197
epoch: 19 loss: 0.00055816834
epoch: 20 loss: 0.0005400749
epoch: 21 loss: 0.00052409666
epoch: 22 loss: 0.0005095422
epoch: 23 loss: 0.0004959964
epoch: 24 loss: 0.00048388544
epoch: 25 loss: 0.0004726796
epoch: 26 loss: 0.00046252742
epoch: 27 loss: 0.0004533096
epoch: 28 loss: 0.00044504667
epoch: 29 loss: 0.00043768054
epoch: 30 loss: 0.0004309544
epoch: 31 loss: 0.0004250586
epoch: 32 loss: 0.0004197291
epoch: 33 loss: 0.00041498753
epoch: 34 loss: 0.00041

In [15]:
v = np.array([0.5, 0.4, 0.3])
training.model.model((v).reshape(1, 3), training = True)

<tf.Tensor: shape=(1, 4), dtype=float32, numpy=array([[1.0155383 , 0.7373393 , 0.93629134, 0.92526263]], dtype=float32)>

In [16]:
#Z = []
#for i in range(40):
#    for j in range(40):
#        for k in range(40):
#            Z.append(training.model.model(np.array([i/20-1,j/20-1,k/20-1]).reshape([1, 3]).astype("float32"), training = True))
            
#result_vals= np.array(Z)[:, :, 0].reshape(40, 40, 40)


In [19]:
results = training.model.model(points, training = True)
#print(results)
results_val = np.array(results)[:, 0]
results_val = results_val[:, np.newaxis] /1.5 + 1
colors = np.tile([1.0, 1.0, 1.0], (results.shape[0], 1))
colors = colors * results_val
print(min(results_val))
print(max(results_val))

[-0.02355039]
[2.0179176]


In [ ]:
ps.init()
ps.set_ground_plane_mode("none")

# register mesh and callback
ps.register_surface_mesh("mesh", positions, indices)

#ps_grid = ps.register_volume_grid("sample grid", (40, 40, 40), (-1., -1., -1.), (1., 1., 1.))
#ps_grid.add_scalar_quantity("prediction result", result_vals, defined_on='nodes')
ps.register_point_cloud("volume",points).add_color_quantity("color", colors)
ps.show()